In [1]:
# 1. Imports
import os
import cv2
import numpy as np
from tqdm import tqdm
from google.colab import drive

# 2. Setup Drive and Pathing
drive.mount('/content/drive', force_remount=True)

# Using your confirmed shortcut path
source_folder = "/content/drive/MyDrive/247C Project/SF_retrofitted_streetview_images/custom_segmented_crops_dash"
uniform_folder = "/content/drive/MyDrive/247C Project/SF_retrofitted_streetview_images/uniform_standardized_inventory_dash"

if not os.path.exists(uniform_folder):
    os.makedirs(uniform_folder)

# 3. Letterboxing Function (Prevents stretching)
def resize_with_padding(img, expected_size):
    img_h, img_w = img.shape[:2]
    res_w, res_h = expected_size

    # Calculate scaling factor
    img_aspect = img_w / img_h
    res_aspect = res_w / res_h

    if img_aspect > res_aspect: # Image is wider than target
        new_w = res_w
        new_h = int(res_w / img_aspect)
    else: # Image is taller than target
        new_h = res_h
        new_w = int(res_h * img_aspect)

    # Resize the building facade
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Create a black background square
    canvas = np.zeros((res_h, res_w, 3), dtype=np.uint8)

    # Center the building on the canvas
    x_offset = (res_w - new_w) // 2
    y_offset = (res_h - new_h) // 2
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

    return canvas

# 4. Processing Loop
all_crops = [f for f in os.listdir(source_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"🚀 Standardizing {len(all_crops)} building crops for CEE 247C...")

for filename in tqdm(all_crops):
    try:
        img_path = os.path.join(source_folder, filename)
        img = cv2.imread(img_path)

        if img is not None:
            # Standardize to 640x640
            standardized_img = resize_with_padding(img, (640, 640))

            # Save to the new folder
            cv2.imwrite(os.path.join(uniform_folder, f"std_{filename}"), standardized_img)
    except Exception as e:
        continue

print(f"🏁 DONE! Uniform building inventory ready in: {uniform_folder}")

Mounted at /content/drive
🚀 Standardizing 4892 building crops for CEE 247C...


100%|██████████| 4892/4892 [03:35<00:00, 22.70it/s]

🏁 DONE! Uniform building inventory ready in: /content/drive/MyDrive/247C Project/SF_retrofitted_streetview_images/uniform_standardized_inventory_dash
